# Punto 3: Problema de la Mochila 0/1 (0/1 Knapsack Problem)
Conjunto de Problemas: Pisinger Benchmark - Uncorrelated, 50 ítems

Estudiantes:
* Lina Sofía Amaya Rodríguez
* Laura Valentina Gonzalez Rincon
* Camilo Londoño Moreno

Universidad Nacional de Colombia - Optimización 2026-1

In [32]:
!pip install pandas
!pip install gurobipy

In [46]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [47]:
FILE_NAME = "./Knapsack_50items.txt"

# Se abre el archivo especificado en FILE_NAME
# Tiene que tener la misma estructura de los test de Pisinger
with open(FILE_NAME) as f:
    nombre = f.readline().strip()
    n = int(f.readline().split()[1])
    capacidad = int(f.readline().split()[1])
    z = int(f.readline().split()[1])
    f.readline()  # Tiempo

print(f"Nombre del test: {nombre}")
print(f"Tamaño (N): {n}")
print(f"Capacidad: {capacidad}")
print(f"Solución Esperada (Z): {z}")

Nombre del test: knapPI_1_50_1000_1
Tamaño (N): 50
Capacidad: 995
Solución Esperada (Z): 8373


In [48]:
# Dataframe de pandas para guardar los datos de los items
datos = pd.read_csv(
    FILE_NAME,
    sep=r"\s+",
    skiprows=5,
    header=None,
    names=["id", "beneficio", "peso", "value"]
)

datos.head()

,id,beneficio,peso,value
0,1,94,485,0
1,2,506,326,0
2,3,416,248,0
3,4,992,421,0
4,5,649,322,0


In [49]:
# Se crea el modelo con Guroby para solucionarlo
m = gp.Model(nombre)

# Se agregan las variables
x = m.addVars(n, vtype=GRB.BINARY, name="x")

# Se establece la función objetivo
m.setObjective(
    gp.quicksum(
        datos.loc[i, "beneficio"] * x[i]
        for i in range(n)
    ),
    GRB.MAXIMIZE
)

# Se añade las restricciones
m.addConstr(
    gp.quicksum(
        datos.loc[i, "peso"] * x[i]
        for i in range(n)
    ) <= capacidad
)

<gurobi.Constr *Awaiting Model Update*>

In [50]:
m.optimize()
# Guardar modelo
m.write("knapsack.lp")

# Guardar solución
m.write("knapsack.sol")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1 rows, 50 columns and 50 nonzeros (Max)
Model fingerprint: 0x103bfc60
Model has 50 linear objective coefficients
Variable types: 0 continuous, 50 integer (50 binary)
Coefficient statistics:
  Matrix range     [9e+00, 1e+03]
  Objective range  [7e+00, 1e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+03]

Found heuristic solution: objective 2515.0000000
Presolve removed 1 rows and 50 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 2: 8373 2515 

Optimal solution found (tolerance 1.00e-04)
Best objective 8.373000000000e+03, best bound 8.373000000000e+03, gap 

In [51]:
print("Óptimo Gurobi:", m.ObjVal)
print("Óptimo archivo:", z)

Óptimo Gurobi: 8373.0
Óptimo archivo: 8373


In [52]:
print("Items seleccionados:")
for i in range(n):
    if x[i].X > 0:
        print(
            f"id: {datos.iloc[i]['id']}, "
            f"valor: {datos.iloc[i]['beneficio']}, "
            f"peso: {datos.iloc[i]['peso']}"
        )

Items seleccionados:
id: 7, valor: 457, peso: 43
id: 11, valor: 791, peso: 9
id: 13, valor: 667, peso: 122
id: 14, valor: 598, peso: 94
id: 24, valor: 700, peso: 72
id: 26, valor: 874, peso: 138
id: 31, valor: 997, peso: 199
id: 33, valor: 908, peso: 97
id: 38, valor: 931, peso: 70
id: 39, valor: 726, peso: 98
id: 49, valor: 724, peso: 29


In [53]:
peso_total = sum(
    datos.loc[i, "peso"]
    for i in range(n)
    if x[i].X > 0
)

print("Peso total:", peso_total)
print(f"Capacidad mochila: {capacidad}")

Peso total: 971
Capacidad mochila: 995


In [54]:
beneficio_total = sum(
    datos.loc[i, "beneficio"]
    for i in range(n)
    if x[i].X > 0
)

print("Beneficio total:", beneficio_total)

Beneficio total: 8373
